In [ ]:
import os
os.environ["PATH"] += ":/usr/local/gromacs/bin"

In [ ]:
import sys
import matplotlib.pyplot as plt
import nglview as ng
import mdtraj as md
import pandas as pd

In [ ]:
%cd ../data

In [ ]:
%rm -rf processing
%rm -rf figures
%rm -rf analysis
%mkdir processing
%mkdir figures
%mkdir analysis

In [ ]:
view = ng.show_structure_file("input/1fjs.pdb")
view

In [ ]:
!grep -v HETATM input/1fjs.pdb > processing/1fjs_protein_tmp.pdb
!grep -v CONECT processing/1fjs_protein_tmp.pdb > processing/1fjs_protein.pdb

In [ ]:
!grep MISSING input/1fjs.pdb

In [ ]:
%cd processing

In [ ]:
!gmx pdb2gmx \
    -f 1fjs_protein.pdb \
    -o 1fjs_processed.gro \
    -water tip3p \
    -ff charmm36

In [ ]:
!cat ../steps/topol.top

In [ ]:
!grep "moleculetype" -A 3 topol_Protein_chain_A.itp

In [ ]:
!grep "moleculetype" -A 3 topol_Protein_chain_L.itp

In [ ]:
! grep "atoms" -A 7 topol_Protein_chain_A.itp

In [ ]:
!grep "bonds" -A 7 topol_Protein_chain_A.itp

In [ ]:
!grep "angles" -A 7 topol_Protein_chain_A.itp

In [ ]:
!grep "dihedrals" -A 7 topol_Protein_chain_A.itp

In [ ]:
!grep "pairs" -A 7  topol_Protein_chain_A.itp

In [ ]:
!grep "posre" -A 3 -B 3 topol_Protein_chain_A.itp

In [ ]:
!grep "position_restraints" -A 7 posre_Protein_chain_A.itp

In [ ]:
!grep "water topology" -A 8 topol.top

In [ ]:
!grep "ions" topol.top

In [ ]:
!grep "system" -A 7 topol.top

In [ ]:
!gmx editconf -f 1fjs_processed.gro -o 1fjs_newbox.gro -c -d 1.0 -bt cubic

In [ ]:
!gmx solvate -cp 1fjs_newbox.gro -cs spc216.gro -o 1fjs_solv.gro -p topol.top

In [ ]:
!tail topol.top

In [ ]:
!touch viz.mdp
!gmx grompp -f viz.mdp -c 1fjs_solv.gro -p topol.top -o topol.tpr
!printf "0\n" | gmx trjconv -f 1fjs_solv.gro -s topol.tpr -o 1fjs_solv_viz.gro -pbc mol -ur compact

In [ ]:
!grep "bonds" -B 20 topol_Protein_chain_A.itp

In [ ]:
!grep "bonds" -B 20 topol_Protein_chain_L.itp

In [ ]:
!touch ions.mdp

In [ ]:
!gmx grompp -f ions.mdp -c 1fjs_solv.gro -p topol.top -o ions.tpr

In [ ]:
!printf "SOL\n" | gmx genion -s ions.tpr -o 1fjs_solv_ions.gro -conc 0.15 -p \
topol.top -pname NA -nname CL -neutral

In [ ]:
!gmx grompp -f viz.mdp -c 1fjs_solv_ions.gro -p topol.top -o topol.tpr
!printf "0\n" | gmx trjconv -f 1fjs_solv_ions.gro -s topol.tpr -o 1fjs_solv_ions_viz.gro -pbc mol -ur compact

In [ ]:
!cat ../input/emin-charmm.mdp

In [ ]:
!gmx grompp -f ../input/emin-charmm.mdp -c 1fjs_solv_ions.gro -p topol.top -o em.tpr

In [ ]:
!gmx mdrun -v -deffnm em -ntmpi 1 -ntomp 1

In [ ]:
!printf "Potential\n0\n" | gmx energy -f em.edr -o ../analysis/potential.xvg -xvg none

In [ ]:
df = pd.read_csv('../analysis/potential.xvg', sep='\\s+', header=None, names=['Potential energy'])
ax = df.plot(xlabel = 'Step', ylabel = 'Energia Potencial (kJ/mol)', title = "Potencial Energético durante Minimização")
fig = ax.get_figure()
fig.savefig('../figures/Potential_Energy.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
!gmx grompp -f ../input/nvt-charmm.mdp -c em.gro -r em.gro -p topol.top -o nvt.tpr 
!gmx mdrun -ntmpi 1 -ntomp 8 -v -deffnm nvt

In [ ]:
!echo "Temperature" | gmx energy -f nvt.edr -o ../analysis/temperature.xvg -xvg none -b 20

In [ ]:
df = pd.read_csv('../analysis/temperature.xvg', sep='\\s+', header=None, names=['Temperature'])
ax = df.plot(xlabel = 'Tempo (ps)', ylabel = 'Temperatura (K)', title = "Temperatura durante Equilibração NVT")
fig = ax.get_figure()
fig.savefig('../figures/Temperature_Energy.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
!gmx grompp -f ../input/npt-charmm.mdp -c nvt.gro -r nvt.gro -t nvt.cpt -p topol.top -o npt.tpr

In [ ]:
!gmx mdrun -ntmpi 1 -ntomp 8 -v -deffnm npt

In [ ]:
!echo "Pressure" | gmx energy -f npt.edr -o ../analysis/pressure.xvg -xvg none

In [ ]:
df = pd.read_csv('../analysis/pressure.xvg', sep='\\s+', header=None, names=['Pressure'])
ax = df.plot(xlabel = 'Tempo (ps)', ylabel = 'Pressão (bar)', title = "Pressão durante Equilibração NPT")
fig = ax.get_figure()
fig.savefig('../figures/Pressure_Energy.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
!echo "Density" | gmx energy -f npt.edr -o ../analysis/density.xvg -xvg none

In [ ]:
df = pd.read_csv('../analysis/density.xvg', sep='\\s+', header=None, names=['Density'])
ax = df.plot(xlabel = 'Tempo (ps)', ylabel = 'Densidade (kg/m^3)', title = "Densidade durante Equilibração NPT")
fig = ax.get_figure()
fig.savefig('../figures/Density_Energy.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
!gmx grompp -f ../input/md-charmm.mdp -c npt.gro -t npt.cpt -p topol.top -o md.tpr

In [ ]:
!gmx mdrun -ntmpi 1 -ntomp 8 -v -deffnm md 

In [ ]:
!gmx trjconv -h

In [ ]:
!printf "1" | gmx trjconv -s md.tpr -f md.xtc -o md_whole.xtc -pbc whole
!printf "1" | gmx trjconv -s md.tpr -f md_whole.xtc -o md_nojump.xtc -pbc nojump
!printf "1\n1" | gmx trjconv -s md.tpr -f md_nojump.xtc -o md_mol.xtc -center -pbc mol

In [ ]:
!printf "1\n" | gmx mindist -s md.tpr -f md_mol.xtc -pi -od ../analysis/mindist.xvg 


In [ ]:
!printf "4\n1\n" | gmx rms -s em.tpr -f md_mol.xtc -o ../analysis/rmsd_xray.xvg -tu ns -xvg none

In [ ]:
df = pd.read_csv('../analysis/rmsd_xray.xvg', sep='\\s+', header=None, names=['RMSD'])
ax = df.plot(xlabel = 'Tempo (ns)', ylabel = 'RMSD (nm)', title = "Root Mean Square Deviation (RMSD) da Proteína em Produção")
fig = ax.get_figure()
fig.savefig('../figures/RMSD_xray.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
!echo "1" | gmx gyrate -f md_mol.xtc -s md.tpr -o ../analysis/gyrate.xvg -xvg none

In [ ]:
df = pd.read_csv('../analysis/gyrate.xvg', sep='\\s+', header=None, names=['Rg'], usecols=[1])
ax = df.plot(xlabel = 'Tempo (ps)', ylabel = 'Raio de Giração (nm)', title = "Raio de Giração (Rg) da Proteína em Produção")
fig = ax.get_figure()
fig.savefig('../figures/Gyrate_rg.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
!printf "1\n" | gmx rmsf -s md.tpr -f md_mol.xtc -o ../analysis/rmsf.xvg -res -xvg none

In [ ]:
df = pd.read_csv('../analysis/rmsf.xvg', sep='\\s+', header=None, names=['RMSF'])
ax = df.plot(xlabel = 'Resíduo', ylabel = 'RMSF (nm)', title = "Root Mean Square Fluctuation (RMSF) da Proteína em Produção")
fig = ax.get_figure()
fig.savefig('../figures/RMSF.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
!gmx sasa -s md.tpr -f md_mol.xtc -o ../analysis/sasa.xvg -surface 'group "Protein"' -output 'group "Protein"' -tu ns -xvg none

In [ ]:
df = pd.read_csv('../analysis/sasa.xvg', sep='\\s+', header=None, names=['SASA'])
ax = df.plot(xlabel = 'Tempo (ns)', ylabel = 'SASA (nm²)', title = "Área de Superfície Acessível ao Solvente (SASA) da Proteína em Produção")
fig = ax.get_figure()
fig.savefig('../figures/SASA.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
traj = md.load('md_mol.xtc', top='md.gro')
protein = traj.atom_slice(traj.topology.select('protein'))
dssp = md.compute_dssp(protein, simplified=True)
pd.DataFrame({'Tempo': traj.time / 1000, 'Alpha Helix': (dssp == 'H').mean(axis=1)}).to_csv('../analysis/alpha_helix.xvg', sep=' ', header=False, index=False)
pd.DataFrame({'Tempo': traj.time / 1000, 'Beta Sheet': (dssp == 'E').mean(axis=1)}).to_csv('../analysis/beta_sheet.xvg', sep=' ', header=False, index=False)
pd.DataFrame({'Tempo': traj.time / 1000, 'Coils': (dssp == 'C').mean(axis=1)}).to_csv('../analysis/coils.xvg', sep=' ', header=False, index=False)

In [ ]:
df = pd.read_csv('../analysis/alpha_helix.xvg', sep='\\s+', header=None, names=['Alpha Helix'])
ax = df.plot(xlabel = 'Tempo (ns)', ylabel = 'Fração de Resíduos', title = "Fração de Hélice Alfa da Proteína em Produção")
fig = ax.get_figure()
fig.savefig('../figures/Alpha_Helix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df = pd.read_csv('../analysis/beta_sheet.xvg', sep='\\s+', header=None, names=['Beta Sheet'])
ax = df.plot(xlabel = 'Tempo (ns)', ylabel = 'Fração de Resíduos', title = "Fração de Folha Beta da Proteína em Produção")
fig = ax.get_figure()
fig.savefig('../figures/Beta_Sheet.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df = pd.read_csv('../analysis/coils.xvg', sep='\\s+', header=None, names=['Coils'])
ax = df.plot(xlabel = 'Tempo (ns)', ylabel = 'Fração de Resíduos', title = "Fração de Coils da Proteína em Produção")
fig = ax.get_figure()
fig.savefig('../figures/Coils.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
!printf "h\nq\n" | gmx make_ndx -f nvt.tpr -o

In [ ]:
!printf "splitch 1\nq\n" | gmx make_ndx -f nvt.tpr -o chains_make_ndx.ndx

In [ ]:
!printf "group "Protein" and mol 1\ngroup "Protein" and mol 2" | gmx select -s nvt.tpr -on chains_select.ndx

In [ ]:
!printf "17\n18\n"| gmx hbond -f md.xtc -s md.tpr -n chains_make_ndx.ndx -num ../analysis/hbnum_ndx.xvg -xvg none

In [ ]:
!printf "0\n1\n"| gmx hbond -f md.xtc -s md.tpr -n chains_select.ndx -num ../analysis/hbnum.xvg -xvg none

In [ ]:
df = pd.read_csv('../analysis/hbnum_ndx.xvg', sep='\\s+', header=None, names=['H-bonds'])
ax = df.plot(xlabel = 'Tempo (ps)', ylabel = 'Ligações de Hidrogênio', title = "Número de Ligações de Hidrogênio entre Cadeias")
fig = ax.get_figure()
fig.savefig('../figures/hbonds.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
!gmx report-methods -s md.tpr